# Import Onda A-mode measurements

Run all cells to load the `A-mode*` folders in `data/Measurements - Onda hydrophone data - 2026`.
Requires pandas and an IPython/Jupyter Python kernel. Works when launched from the project root or `src`.

- `average_df`: saved averaged readings, including the detrended amplitude and A-mode envelope.
- `pulses_df`: individual pulse readings, identified by `measurement` and `pulse_index`.
- `metadata_df`: acquisition metadata from each file's comment header.

Each waveform row represents one time sample. Original CSV column names and units are preserved (seconds and volts). The saved average is detrended, whereas pulse amplitudes are the recorded waveforms; they need not match even for a single pulse. Voltage columns are retained. The calibration cells add pressure columns in Pa using the sensitivity at a user-specified frequency for each recording.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Locate the project when the kernel starts in the repository or src directory.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "Measurements - Onda hydrophone data - 2026").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start the notebook from the project root or src directory.")

measurement_root = project_root / "data" / "Measurements - Onda hydrophone data - 2026"
measurement_dirs = sorted(path for path in measurement_root.glob("A-mode*") if path.is_dir())
if not measurement_dirs:
    raise FileNotFoundError(f"No A-mode folders found in {measurement_root}")

[p.name for p in measurement_dirs]

In [ ]:
def read_a_mode_csv(path):
    """Read waveform samples and retain the acquisition header separately."""
    metadata = {}
    with path.open(encoding="utf-8-sig") as stream:
        for line in stream:
            if not line.startswith("#"):
                break
            key, separator, value = line[1:].strip().partition(":")
            if separator:
                metadata[key.strip()] = value.strip()

    waveform = pd.read_csv(path, comment="#", encoding="utf-8-sig")
    required = (
        {"Time_s", "Average_Detrended_Amplitude_V", "A_Mode_Envelope_V"}
        if path.name == "a_scan_average.csv" else {"Time_s", "Amplitude_V"}
    )
    missing = required - set(waveform.columns)
    if missing:
        raise ValueError(f"{path}: missing columns {sorted(missing)}")
    for column in required:
        waveform[column] = pd.to_numeric(waveform[column], errors="raise")
    waveform.insert(0, "measurement", path.parent.name)
    waveform.insert(1, "source_file", str(path.relative_to(measurement_root)))
    metadata.update(measurement=path.parent.name,
                    source_file=str(path.relative_to(measurement_root)))
    return waveform, metadata


average_frames, pulse_frames, metadata_rows = [], [], []
for folder in measurement_dirs:
    average_path = folder / "a_scan_average.csv"
    pulse_paths = sorted(folder.glob("a_scan_pulse_*.csv"),
                         key=lambda path: int(path.stem.rsplit("_", 1)[1]))
    if not average_path.is_file() or not pulse_paths:
        raise FileNotFoundError(f"Expected average and per-pulse CSV files in {folder}")

    frame, metadata = read_a_mode_csv(average_path)
    average_frames.append(frame)
    metadata_rows.append({**metadata, "reading_type": "average"})
    for path in pulse_paths:
        frame, metadata = read_a_mode_csv(path)
        pulse_index = int(path.stem.rsplit("_", 1)[1])
        frame.insert(2, "pulse_index", pulse_index)
        pulse_frames.append(frame)
        metadata_rows.append({**metadata, "pulse_index": pulse_index,
                              "reading_type": "pulse"})

average_df = pd.concat(average_frames, ignore_index=True)
pulses_df = pd.concat(pulse_frames, ignore_index=True)
metadata_df = pd.DataFrame(metadata_rows)
metadata_df["pulse_index"] = pd.to_numeric(metadata_df["pulse_index"]).astype("Int64")

In [ ]:
# Sample counts and previews of the imported DataFrames.
display(average_df.groupby("measurement", sort=False).size().rename("average_samples").to_frame())
display(pulses_df.groupby(["measurement", "pulse_index"], sort=False).size().rename("pulse_samples").to_frame())
display(average_df.head())
display(pulses_df.head())
display(metadata_df)

## Hydrophone calibration: voltage to pressure

Edit `measurement_frequencies_mhz` below to specify the measurement frequency in **MHz** for each recording. The calibration file is read from that recording's folder (with or without a `.txt` extension).

Use the table's **Sens(V/Pa)** column directly. For a frequency between table entries, linearly interpolate sensitivity between the closest lower and higher frequencies. Frequencies outside the calibrated range raise an error.

Pressure is `amplitude_V / sensitivity_V_per_Pa`. This applies one sensitivity at the specified frequency to the whole waveform, including the envelope; it is a single-frequency conversion, without broadband frequency-response correction. The calibration file includes the AG-2010 preamplifier, so no extra amplifier gain factor is applied.

In [ ]:
# USER SETTINGS: measurement frequencies in MHz (edit these values).
measurement_frequencies_mhz = {
    "A-mode_2MHz_focus_10Vpp_20cycles": 2.0,
    "A-mode_7p37MHz_focus_10Vpp_20cycles": 7.37,
}
calibration_basename = "HGL0400-2875_AG-2010-1384-20_xx_OndaCombineCal_20230621-Reza-Oct-2026"

In [ ]:
import io
import numpy as np


def read_calibration(path):
    lines = path.read_text(encoding="utf-8-sig").splitlines()
    header_end = next((i for i, line in enumerate(lines)
                       if line.strip().startswith("HEADER_END")), None)
    if header_end is None:
        raise ValueError(f"Missing HEADER_END marker in {path}")
    table = pd.read_csv(
        io.StringIO("\n".join(lines[header_end + 1:])),
        sep=r"\s+", comment="#", header=None,
        names=["frequency_mhz", "sensitivity_db", "sensitivity_v_per_pa",
               "sensitivity_v2_cm2_per_w"],
    )
    table = table.apply(pd.to_numeric, errors="raise").sort_values("frequency_mhz").reset_index(drop=True)
    if (table.empty or not np.isfinite(table.to_numpy()).all()
            or table["frequency_mhz"].duplicated().any()
            or (table["sensitivity_v_per_pa"] <= 0).any()):
        raise ValueError(f"Invalid calibration table in {path}")
    return table


def sensitivity_at_frequency(table, frequency_mhz):
    frequency_mhz = float(frequency_mhz)
    frequencies = table["frequency_mhz"].to_numpy()
    sensitivities = table["sensitivity_v_per_pa"].to_numpy()
    if not np.isfinite(frequency_mhz) or not frequencies[0] <= frequency_mhz <= frequencies[-1]:
        raise ValueError(f"Frequency must be within {frequencies[0]}–{frequencies[-1]} MHz")
    upper = int(np.searchsorted(frequencies, frequency_mhz))
    if frequencies[upper] == frequency_mhz:
        return float(sensitivities[upper]), float(frequencies[upper]), float(frequencies[upper])
    lower = upper - 1
    fraction = (frequency_mhz - frequencies[lower]) / (frequencies[upper] - frequencies[lower])
    sensitivity = sensitivities[lower] + fraction * (sensitivities[upper] - sensitivities[lower])
    return float(sensitivity), float(frequencies[lower]), float(frequencies[upper])


calibration_tables = {}
calibration_rows = []
for folder in measurement_dirs:
    if folder.name not in measurement_frequencies_mhz:
        raise ValueError(f"Specify a frequency in MHz for {folder.name}")
    candidates = [folder / (calibration_basename + ".txt"), folder / calibration_basename]
    calibration_path = next((p for p in candidates if p.is_file()), None)
    if calibration_path is None:
        raise FileNotFoundError(f"Calibration file missing in {folder}")
    table = read_calibration(calibration_path)
    calibration_tables[folder.name] = table
    frequency = measurement_frequencies_mhz[folder.name]
    sensitivity, lower, upper = sensitivity_at_frequency(table, frequency)
    calibration_rows.append({
        "measurement": folder.name, "frequency_mhz": float(frequency),
        "sensitivity_v_per_pa": sensitivity,
        "lower_frequency_mhz": lower, "upper_frequency_mhz": upper,
        "interpolated": lower != upper,
        "calibration_file": str(calibration_path.relative_to(measurement_root)),
    })

calibration_df = pd.DataFrame(calibration_rows)
sensitivity_by_measurement = calibration_df.set_index("measurement")["sensitivity_v_per_pa"]
for frame, columns in [
    (average_df, {"Average_Detrended_Amplitude_V": "Average_Detrended_Amplitude_Pa",
                  "A_Mode_Envelope_V": "A_Mode_Envelope_Pa"}),
    (pulses_df, {"Amplitude_V": "Amplitude_Pa"}),
]:
    frame["sensitivity_v_per_pa"] = frame["measurement"].map(sensitivity_by_measurement)
    for voltage_column, pressure_column in columns.items():
        frame[pressure_column] = frame[voltage_column] / frame["sensitivity_v_per_pa"]

display(calibration_df)
display(average_df.head(), pulses_df.head())

In [ ]:
# Example: select one recording and its first pulse for further processing.
measurement = measurement_dirs[0].name
average = average_df.loc[average_df["measurement"].eq(measurement)].copy()
pulse = pulses_df.loc[
    pulses_df["measurement"].eq(measurement) & pulses_df["pulse_index"].eq(1)
].copy()
display(average.head(), pulse.head())

In [ ]:
# Overlay the selected pulse and saved average in Pa.
import plotly.graph_objects as go

fig = go.Figure()
fig.add_scatter(x=pulse["Time_s"], y=pulse["Amplitude_Pa"],
                mode="lines", name="Pulse 1", opacity=0.7)
fig.add_scatter(x=average["Time_s"], y=average["Average_Detrended_Amplitude_Pa"],
                mode="lines", name="Average (detrended)")
fig.add_scatter(x=average["Time_s"], y=average["A_Mode_Envelope_Pa"],
                mode="lines", name="Envelope")
fig.update_layout(title=measurement, xaxis_title="Time (s)",
                  yaxis_title="Pressure (Pa)", template="plotly_white",
                  hovermode="x unified")
fig.show()